# Tutorial 2: Text Generation

In our previous session, we constructed a complete training pipeline in PyTorch, mastering the fundamentals of processing fixed-size data. Now, we will extend these skills to one of the most dynamic and impactful domains in AI: **Text Generation**.

The primary challenge shifts from classifying static inputs to modeling **sequential, variable-length data**. This tutorial will guide you through the entire lifecycle of a modern language model. We will start with the core principles of representing text and training a sequence-aware model like the Transformer. Subsequently, we will dive into the inference phase, exploring the strategies that control what the model writes (Decoding Strategies) and the optimizations that determine how fast it writes (KV Cache).


## Part 1: Principle of LLMs

In this section, we will construct the core engine of a language model. Our goal is to understand how a model learns to "write" by predicting the next word in a sequence. This involves translating raw text into a format the model can understand and implementing the architecture that enables it to learn contextual patterns.

- **Tokenizer**: Learn how to convert raw text into numerical tokens and back, creating the vocabulary that defines our model's world.
- **Teacher Forcing**: Understand the fundamental training strategy for autoregressive models, where we guide the model with the correct sequence to accelerate learning.
- **Transformer**: Implement the essential components of the Transformer architecture, specifically the self-attention mechanism with a **causal mask**, which allows the model to generate text without seeing into the future.

### **1.1 Tokenizer**

Before a Large Language Model (LLM) can understand or generate a single word, the raw text must be translated into a mathematical format the computer can process. This translation is the job of the **Tokenizer**.

In Generative AI, we cannot feed strings directly into neural networks. Instead, we must break text down into discrete units called **tokens** and convert them into numerical sequences. If the model is the engine, the tokenizer is the fuel injector—it decides exactly how data is prepared, determining the model's vocabulary size, its ability to handle unknown words, and ultimately, its efficiency in generating coherent language.

#### **1.1.1 From Characters to Subwords**

So, how do we actually transform a sentence like "I am learning GenAI" into a sequence of numbers? We use an algorithm called **Byte-Pair Encoding**, or **BPE**. This is currently the industry standard for almost all modern Large Language Models.

The core idea of BPE is **compression**. We want to represent the text using the most efficient vocabulary possible. Instead of treating every single character as a token (which makes sequences too long) or every single word as a token (which creates a massive, unmanageable vocabulary), BPE finds a "sweet spot" in the middle: the **subword**.

The BPE algorithm builds its vocabulary iteratively. Here is how it learns:

*   **Initialization with Characters**:
    First, the algorithm looks at your entire training corpus and breaks every word down into individual characters. For example, the word "learning" starts as the sequence: `l, e, a, r, n, i, n, g`. At this stage, our vocabulary only consists of basic ASCII characters.

*   **Counting Frequency**:
    Next, the algorithm scans the corpus to count how often adjacent pairs of tokens appear together. It looks for the most frequent pair. In English, you might see "h" and "e" appearing together very often to form "he", or "i" and "n" forming "in".

*   **Merging the Most Frequent Pair**:
    Once the most frequent pair is identified, BPE **merges** them into a single new token. If "i" and "n" is the most frequent pair, they are fused into a new token "in". Now, "learning" becomes `l, e, a, r, n, in, g`. The vocabulary has just grown by one.

*   **Iteration**:
    This process repeats thousands of times. We count pairs again, find the new winner, and merge. Eventually, common whole words like "the" or "apple" become single tokens, while rare words remain split into smaller subword chunks. This ensures that the model can process common words efficiently while still being able to construct rare words from their parts.

In [ ]:
from collections import defaultdict

# 1. Initialization: Start with a corpus split into individual characters
# We use a small list of words to demonstrate the "learning" process.
corpus = ["learning", "leaning", "earn"]
# Represent each word as a list of tokens (initially characters)
vocab_splits = [list(word) for word in corpus]

print(f"Start: {vocab_splits}\n")

# Run BPE for a few iterations to see subwords form
num_merges = 5

for step in range(num_merges):
    # 2. Counting Frequency: Scan corpus for adjacent pairs
    pairs = defaultdict(int)
    for word in vocab_splits:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += 1

    # If no pairs exist, we can't merge anymore
    if not pairs:
        break

    # 3. Merging the Most Frequent Pair: Find the "winner"
    best_pair = max(pairs, key=pairs.get)
    new_token = "".join(best_pair)

    print(f"Merge {step + 1}: Found frequent pair {best_pair} -> Merging into '{new_token}'")

    # 4. Iteration: Update the corpus by replacing the pair with the new token
    new_splits = []
    for word in vocab_splits:
        new_word = []
        i = 0
        while i < len(word):
            # Check if current and next token match the best pair
            if i < len(word) - 1 and (word[i], word[i+1]) == best_pair:
                new_word.append(new_token) # Merge
                i += 2                     # Skip next token
            else:
                new_word.append(word[i])   # Keep existing
                i += 1
        new_splits.append(new_word)

    vocab_splits = new_splits
    print(f"State: {vocab_splits}\n")

# Final result: Common subwords like 'ear', 'ni', 'ng' are formed


#### **1.1.2 Vocabulary and Mapping**

Once BPE has determined which subwords make up our language, we lock this set of subwords into a fixed list called the **Vocabulary**.

The Vocabulary acts essentially as a massive lookup table or a dictionary. The size of this vocabulary is a critical hyperparameter—usually ranging from 30,000 to over 100,000 entries depending on the model. The primary job here is **Mapping**: assigning a unique integer ID to every single token in that list. When we feed text into the model, we represent "Generative" not as a string, but perhaps as the integer ID `15486`.

 However, natural language isn't just about the words we speak; it's also about structure. To handle this, the mapping must include **Special Tokens**. These are tokens that don't represent actual words, but rather serve as control signals for the neural network.

While the specific implementation varies wildly between architectures, you will frequently encounter a few standard types in the mapping:

*   **Padding Token (`[PAD]`):** This is purely a placeholder used to fill up empty space so that all sequences in a batch have the same length.
*   **Start and End Tokens (`[BOS]`, `[EOS]`):** "Beginning of Sentence" and "End of Sentence" tokens mark the boundaries of a thought. These are crucial in generation tasks so the model knows when to stop talking.
*   **Separation and Classification Tokens (`[SEP]`, `[CLS]`):** These are often used to separate two different sentences in the same input, or to mark a specific position where the model should aggregate a summary of the entire input.

The most important takeaway here is that this mapping is a strict **contract**. If you train a model where ID `500` means "apple," you simply cannot use a different tokenizer during inference that maps "apple" to ID `600`. The weights of this model are closely tied to these specific integer IDs.

In [ ]:
# 1. Define Special Tokens (Control Signals)
# These are structural markers essential for the model's lifecycle.
# [PAD]: 0, [BOS]: 1, [EOS]: 2, [UNK]: 3
SPECIAL_TOKENS = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]

# 2. Simulate "Learned" Subwords from BPE (Step 1.1.1)
# In reality, this list contains 30k-100k tokens derived from a massive corpus.
learned_subwords = ["Gener", "ative", "AI", "model", "ing"]

# 3. Construct the Vocabulary (The "Contract")
# We lock the order to ensure ID consistency.
vocab_list = SPECIAL_TOKENS + learned_subwords
token2id = {token: idx for idx, token in enumerate(vocab_list)}
id2token = {idx: token for idx, token in enumerate(vocab_list)}

print(f"Vocabulary Size: {len(token2id)}")
print(f"ID for 'Gener': {token2id['Gener']}") # e.g., 4

# 4. The Mapping Process (Encoding)
# Assume the tokenizer (1.1.1) has already split "Generative AI" into subwords.
raw_sequence = ["Gener", "ative", "AI"]

# We wrap the sequence with [BOS] and [EOS] to mark boundaries.
input_ids = [token2id["[BOS]"]]
for token in raw_sequence:
    # Fallback to [UNK] if the token is not in our fixed vocabulary
    input_ids.append(token2id.get(token, token2id["[UNK]"]))
input_ids.append(token2id["[EOS]"])

# Result: A tensor of integers ready for the Embedding Layer
print(f"Mapped Sequence: {input_ids}")
# Output example: [1, 4, 5, 6, 2] -> [BOS, Gener, ative, AI, EOS]

# 5. Reverse Mapping (Decoding)
# Converting IDs back to text to verify the contract holds.
decoded_tokens = [id2token[idx] for idx in input_ids]
print(f"Decoded: {decoded_tokens}")


#### **1.1.3 Padding and Truncation**

Now that we have our token IDs, we encounter a practical engineering problem. Neural networks process computation in batches for efficiency. The underlying hardware requires these batches to be perfect rectangles, meaning they must be tensors where every row has the exact same length.

However, real-world sentences vary wildly in length. To fix this, we use **Padding** and **Truncation**. If a sequence is shorter than our target length, we append the special Padding Token we discussed earlier to fill the empty space. Conversely, if a sequence is too long, we simply cut it off, or truncate it, to fit the maximum allowable context window.

But there is a catch. We cannot let the model treat these padding tokens as actual data, or it will try to find meaning in empty placeholders. This is where the **Attention Mask** comes in. It is a binary tensor that runs parallel to your input IDs. It typically assigns a "1" for real tokens and a "-inf" for padding tokens. This signal acts as a filter, telling the model's self-attention mechanism to mathematically ignore the padding completely during calculation.

While this padded, two-dimensional batching is the standard foundation, I have to briefly mention that it is not the only way. In some advanced training pipelines, we avoid the waste of computing padding. Instead, we concatenate multiple disjoint sentences together into specific, long one-dimensional sequences. But regardless of those optimizations, understanding the relationship between Padding and the Attention Mask is fundamental to building a tiny language model.

In [ ]:
import torch

# Simulate token IDs from a tokenizer (jagged sequences of varying lengths)
raw_batch = [
    [101, 7592, 2003, 102],                  # Length 4: "Hello is [SEP]"
    [101, 2054, 102],                        # Length 3: "What [SEP]" (Too short)
    [101, 2023, 2003, 1037, 2500, 3000, 102] # Length 7: "This is a long... [SEP]" (Too long)
]

MAX_LEN = 5      # The fixed context window size
PAD_ID = 0       # The ID representing the empty padding token

input_ids = []
attention_masks = []

for seq in raw_batch:
    # --- 1. Truncation ---
    # If sequence exceeds MAX_LEN, cut it off
    if len(seq) > MAX_LEN:
        seq = seq[:MAX_LEN]

    # --- 2. Padding ---
    # Calculate required padding to reach MAX_LEN
    num_pads = MAX_LEN - len(seq)
    # Append Pad Tokens to the end
    padded_seq = seq + [PAD_ID] * num_pads

    # --- 3. Attention Mask ---
    # Create a binary mask: 1 for real data, 0 for padding.
    # Note: In the Self-Attention layer, '0' positions will be replaced
    # with -inf to result in zero probability after Softmax.
    mask = [1] * len(seq) + [0] * num_pads

    input_ids.append(padded_seq)
    attention_masks.append(mask)

# Convert to Tensors (The "Perfect Rectangles" required by hardware)
batch_inputs = torch.tensor(input_ids)
batch_masks = torch.tensor(attention_masks)

print("Rectangular Input Batch:\n", batch_inputs)
print("\nAttention Mask:\n", batch_masks)


#### **1.1.4 From IDs back to Text**

We have covered how data enters the model; now let's discuss how it comes out. This is the **Decoding** phase.

After the Large Language Model performs its forward pass, it does not output text directly. Instead, it produces a probability distribution over the entire vocabulary. Once we select a specific ID from these probabilities—whether through greedy search or random sampling—we are left with a sequence of integers. We need to translate these IDs back into human-readable strings.

This process is not as simple as just joining words with spaces. Because we used algorithms like BPE, a single word might have been split into multiple subword tokens. For example, if the tokenizer split "learning" into "learn" and "##ing", or perhaps used a special underscore to mark the start of a word, the decoder must recognize these specific patterns. It has to stitch these subwords back together seamlessly to reconstruct the original word, effectively reversing the split operation.

Finally, we have the issue of cleanup. The model generates everything, including the control signals we discussed earlier, like the `[EOS]` (End-of-Sentence) token or potentially some `[PAD]` tokens. In a real-world application, we do not want to show these to the user. To handle this, decoding functions typically offer a parameter like `skip_special_tokens`. This utility automatically filters out the structural artifacts, returning only the clean, natural text content.

In [ ]:
from transformers import AutoTokenizer

# 1. Setup: Load a tokenizer (using BERT here as it clearly demonstrates subword splitting with '##')
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Simulated Model Output: A sequence of integers (IDs) selected from the probability distribution
# Represents: "learning is fun" + [SEP] (End of Sentence) + [PAD] (Padding)
generated_ids = [4083, 2075, 2003, 4569, 102, 0, 0]

# 3. Visualizing the Subwords (The "Split" Issue)
# Notice how "learning" was split into "learn" and "##ing" during tokenization
raw_tokens = tokenizer.convert_ids_to_tokens(generated_ids)
print(f"Raw Tokens: {raw_tokens}")
# Output: ['learn', '##ing', 'is', 'fun', '[SEP]', '[PAD]', '[PAD]']

# 4. Basic Decoding (Stitching Subwords)
# The decoder handles the BPE rules, merging "learn" + "##ing" -> "learning"
# However, structural artifacts ([SEP], [PAD]) remain visible here.
raw_text = tokenizer.decode(generated_ids, skip_special_tokens=False)
print(f"Raw Text:   '{raw_text}'")
# Output: 'learning is fun [SEP] [PAD] [PAD]'

# 5. Clean Decoding (Filtering Artifacts)
# Using `skip_special_tokens=True` removes [SEP] and [PAD], returning natural text.
clean_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"Clean Text: '{clean_text}'")
# Output: 'learning is fun'

### **1.2 Teacher Forcing**

Now that we understand how to convert text into token IDs, we move to the next logical step. How do we actually train a model to produce these sequences?
This brings us to a specific training technique known as **Teacher Forcing**. It is the standard algorithm used to train almost all sequence generation models.

In this section, we will define the crucial difference between how a model behaves during training versus how it behaves when generating text for a user. We will look at how to manipulate our token sequences to create the necessary inputs and labels, and specifically how we structure the data to allow the model to learn efficiently.

#### **1.2.1 Training vs. Inference Strategy**

To understand how we train these models, we first need to look at how they create text during actual usage, or **Inference**.

When you chat with a model, it operates in an **Auto-regressive** mode. The model generates the first word, then we feed that generated word back into the input to generate the second word, and so on. It feeds on its own predictions.

However, if we tried to **train** a fresh model this way, we would face a major problem. Because an untrained model outputs garbage effectively at random, the first word it generates will likely be wrong. If we feed that wrong word back as input for the next step, the model becomes confused. The errors accumulate rapidly, drifting further and further away from the correct sentence. The model would never converge because it is trying to learn from its own mistakes before it knows anything at all.

This is why we use **Teacher Forcing** during the Pre-training phase.

In Teacher Forcing, we do not care what the model *predicted* at the previous step. Instead, effectively, we force the model to look at the **Ground Truth**—the actual correct word from the training data—as the input for the next step. It is like a teacher correcting a student immediately after every single word, ensuring the context remains perfect regardless of the student's answer. This stabilizes training and allows parallel computation.

I should note for clarity that while this is the standard for Pre-training, later stages like **Reinforcement Learning from Human Feedback (RLHF)** actually do switch back to generating its own outputs, or "rollouts." But typically, by that stage, the model is already smart enough that its own generations are meaningful, so the error accumulation issue is no longer catastrophic.


In [ ]:
import torch

# Mock setup: A dummy model and a ground truth sequence (IDs)
# Ground Truth: [BOS, A, B, C]
ground_truth = torch.tensor([[101, 200, 201, 202]])
# A dummy model function that outputs a random token ID
model = lambda x: torch.randint(0, 1000, (1, 1))

# ==========================================
# 1. Inference Strategy (Auto-regressive)
# ==========================================
# During usage, the model feeds on its own predictions.
# This is strictly serial: Step T depends on the output of Step T-1.

current_input = ground_truth[:, :1] # Start with [BOS]

print("--- Inference Flow ---")
for _ in range(3):
    # 1. Predict next token based on current history
    prediction = model(current_input)

    # 2. CRITICAL: The *prediction* becomes the input for the next step
    # If 'prediction' is wrong (garbage), the next step is confused.
    current_input = torch.cat([current_input, prediction], dim=1)


# ==========================================
# 2. Training Strategy (Teacher Forcing)
# ==========================================
# We ignore the model's previous predictions for the history.
# We force the model to see the correct Ground Truth context.

print("\n--- Teacher Forcing Flow ---")

# Conceptually, we correct the input at every step.
# In Transformers, this allows us to process the whole sequence in PARALLEL.
# We simply feed the Ground Truth sequence directly.

# The model predicts the next token for every position simultaneously.
# Even if the model predicts wrong at position 2, position 3 still sees the correct Ground Truth.
outputs = model(ground_truth)

# Note: Specific input/target shifting (e.g., x[:-1] vs x[1:])
# will be handled in Section 1.2.2.


#### **1.2.2 Preparing Input and Target Sequences**

So how do we mathematically Implement this correction mechanism? We do it by creating two slightly shifted versions of our data: the **Input Sequence** and the **Target Sequence**.

When we pre-train a Large Language Model, our data source is often just a massive dump of raw text collected from the internet. This could be Wikipedia articles, Reddit conversations, Github code, or news reports. The model is not explicitly told "this is a question" and "this is an answer." It simply learns to predict the next token in a continuous stream of text.

To set this up for Teacher Forcing, we take a sequence of tokens and perform a **Shift**.

Imagine recent news text says: *"The sky is blue"*.

For the **Input Sequence**, which goes into the model, we include everything *except* the very last token. We want the model to see context. So the input is *"The sky is"*.

For the **Target Sequence**, which is the correct answer key, we want to predict the immediate future. So we take the same sequence but shift it forward by one step. The target contains everything *except* the very first token. So the target is *"sky is blue"*.

This creates a perfect alignment for every position in the sequence. When the model sees *"The"* at position 1, its target at position 1 is *"sky"*. When it sees *"sky"* at position 2, the target is *"is"*.

This slicing operation allows us to calculate the error for every token in the sentence simultaneously in a single forward pass, which makes training extremely efficient.

In [ ]:
import torch

# 1. Raw Text Data (e.g., from the internet)
raw_text = "The sky is blue"

# 2. Tokenization (Recalling Section 1.1)
# We map words to Integer IDs.
# Let's assume: "The"->10, "sky"->11, "is"->12, "blue"->13, <EOS>->99
# We typically add an End-Of-Sentence (EOS) token to signal the text is done.
data = torch.tensor([10, 11, 12, 13, 99])

# 3. The "Shift" Operation
# Input Sequence (X):  Include everything EXCEPT the last token.
# Target Sequence (Y): Include everything EXCEPT the first token.
input_ids  = data[:-1]  # [The, sky, is, blue]
target_ids = data[1:]   # [sky, is, blue, EOS]

# 4. Verification
print(f"Original: {data.tolist()}")
print(f"Inputs:   {input_ids.tolist()}")
print(f"Targets:  {target_ids.tolist()}")

print("\n--- Alignment for Teacher Forcing ---")
# This shows what the model learns at each position simultaneously:
vocab_map = {10: "The", 11: "sky", 12: "is", 13: "blue", 99: "<EOS>"}

for x, y in zip(input_ids, target_ids):
    print(f"Pos {input_ids.tolist().index(x)}: Input '{vocab_map[x.item()]}' \t-> Target '{vocab_map[y.item()]}'")


#### **1.2.3 Cross-Entropy Loss**

We have prepared our inputs and our targets. Now we need a metric to tell the model how well it is performing. In language modeling, this metric is almost exclusively the **Cross-Entropy Loss**.

To understand this, we must look at the data shapes.
When you feed your Input Sequence into the model, the output is a raw numerical score called a **Logit** for every single word in the vocabulary, for every position in the sentence.
Therefore, the shape of the model output is typically a 3-dimensional tensor: `(Batch Size, Sequence Length, Vocabulary Size)`.

However, our Target Sequence is much simpler. It is just the correct integer IDs. Its shape is 2-dimensional: `(Batch Size, Sequence Length)`.

The Cross-Entropy Loss function acts as a bridge between these two shapes. It looks at every position in the sequence individually.
For example, at the first position, the model might assign high scores to a thousand different words. The Loss function looks at the Target ID for that position, checks the probability the model assigned to that specific correct ID, and calculates the penalty. The lower the probability assigned to the correct word, the higher the loss.

There is one important technical detail here regarding the padding we discussed earlier. We do not want the model to learn how to predict "Padding." It is a waste of capacity. Therefore, when defining the loss function in PyTorch, we typically set an argument called `ignore_index` to the ID of our padding token. This instruction ensures that any position containing a pad token contributes zero to the loss and generates no gradients during backpropagation.

In [ ]:
import torch
import torch.nn as nn

# Configuration
BATCH_SIZE = 2
SEQ_LEN = 5
VOCAB_SIZE = 100  # Size of the vocabulary
PAD_ID = 0        # The ID reserved for padding

# 1. Model Output (Logits)
# Shape: (Batch Size, Sequence Length, Vocabulary Size)
# These are raw scores for every word in the vocab at every position.
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

# 2. Target Sequence (Ground Truth IDs)
# Shape: (Batch Size, Sequence Length)
# These are just the correct integer indices.
targets = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))

# Simulate padding: The second sentence is shorter, so the last token is PAD.
targets[1, -1] = PAD_ID

print(f"Logits shape:  {logits.shape}")   # [2, 5, 100]
print(f"Targets shape: {targets.shape}")  # [2, 5]

# 3. Defining the Loss Function
# 'ignore_index' tells PyTorch: "If the target is 0 (PAD), don't calculate loss."
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# 4. Calculating Loss
# PyTorch CrossEntropyLoss expects:
#   Input: (N, C) -> (Total Tokens, Vocab Size)
#   Target: (N)   -> (Total Tokens)
# We flatten the Batch and Sequence dimensions together.
# This treats every token in the batch as an independent classification problem.

loss = criterion(
    logits.view(-1, VOCAB_SIZE),  # Flatten to [10, 100]
    targets.view(-1)              # Flatten to [10]
)

print(f"Calculated Loss: {loss.item():.4f}")
# Gradients will be generated for valid tokens, but 0 for the PAD_ID position.


#### **1.2.4 Exposure Bias**

Before we conclude this section on training principles, we need to address a theoretical flaw in the method we just described. This flaw is known in the field as **Exposure Bias**.

The problem stems from the difference in how the model experiences data during training versus how it experiences data during deployment.

During training with Teacher Forcing, the model is in a very safe environment. It is always guided by the correct history. Even if the model predicts the wrong word at step 3, at step 4 we ignore that mistake and provide the correct history anyway. The model never has to deal with the consequences of its own errors. It is effectively being held by the hand.

However, during inference and testing, the teacher is gone. The model must rely entirely on its own previously generated tokens to predict the future. If the model makes a small mistake at the beginning of a sentence, that mistake acts as input for the next step. This can cause the model to enter a state it has never seen during training, leading to further mistakes. This disconnect is the Exposure Bias.

While standard Large Language Model training usually accepts this risk due to the efficiency benefits of Teacher Forcing, researchers have developed mitigation strategies. One such strategy is **Scheduled Sampling**. In this approach, we gently transition the model. Early in training, we use fully guided Teacher Forcing. As training progresses, we occasionally flip a coin and force the model to use its own predicted token as the next input, rather than the ground truth. This prepares the model for the uncertainty it will face in the real world.

### **1.3 Transformer**

We have explored how to tokenize text and how to train a model using Teacher Forcing. Now we arrive at the model architecture itself: the Transformer.

Since we have already covered the basic building blocks like Embeddings and standard Feed-Forward Networks in previous PyTorch tutorials, we will not repeat them here. Instead, we will focus on the two important components that almost define modern Large Language Models: the Multi-Head Self Attention mechanism and Rotary Positional Embeddings.

These are the specific modifications that allow LLMs to handle complex context and long sequences much better than the original designs from a few years ago. In this section, we will look at how to implement the multiple heads that allow the model to understand in parallel, and the mathematical trick that helps the model understand position through rotation.

#### **1.3.1 Multi-Head Self Attention (MHSA)**

We start with the core engine of the Transformer: the Multi-Head Self Attention (https://arxiv.org/pdf/1706.03762).

You are likely familiar with the standard self-attention mechanism, where a query vector searches for relevant information in a set of key vectors. In a single-head system, the model computes one global attention pattern over the sentence. It produces a single weighted sum for each token.

The Multi-Head variation extends this idea by running several self-attention operations in parallel.

From an implementation perspective, we typically project our input features into Queries, Keys, and Values using one large linear layer. However, before we compute the attention scores, we split these tensors into smaller, independent chunks. Each chunk represents a "Head."

For example, if our model dimension is 512 and we have 8 heads, each head will process vectors of size 64. Crucially, the attention scores—the dot products and softmax operations—are calculated separately within each head. Head 1 calculates its own unique attention pattern, totally independent of Head 2.

A common way to interpret this design is through the concept of **Representation Subspaces**.

By initializing multiple heads randomly, we give the model the opportunity to learn distinct aspects of the language simultaneously. One head might naturally drift towards focusing on local grammatical relationships, like which adjective modifies which noun. Another head might end up capturing longer-range dependencies, such as connecting a pronoun back to the name mentioned three sentences ago.

While a single large head could theoretically learn complex patterns, splitting the computation into multiple heads encourages the model to develop a diverse set of perspectives on the same input sequence. After each head finishes its work, we concatenate their outputs back together, combining these different viewpoints into a single, rich representation.

In [ ]:
import torch
import torch.nn.functional as F

# --- Setup Dimensions ---
B, T, C = 2, 10, 512   # Batch=2, Seq_Len=10, Model_Dim=512
n_heads = 8
head_dim = C // n_heads # 512 / 8 = 64

# Simulate Q, K, V (Assume these are already projected via a Linear layer)
q = torch.randn(B, T, C)
k = torch.randn(B, T, C)
v = torch.randn(B, T, C)

# ==========================================
# 1. Single-Head Perspective (Global)
# ==========================================
# The model treats the entire embedding (C=512) as one vector.
# Attention is computed globally.
out_single = F.scaled_dot_product_attention(q, k, v)

print(f"Single-Head Input: {q.shape}")       # [2, 10, 512]
print(f"Single-Head Output: {out_single.shape}\n")

# ==========================================
# 2. Multi-Head Perspective (Subspaces)
# ==========================================
# We split 'C' into 'n_heads' independent chunks to create subspaces.

# A. Reshape: [B, T, C] -> [B, T, n_heads, head_dim]
q_mh = q.view(B, T, n_heads, head_dim)
k_mh = k.view(B, T, n_heads, head_dim)
v_mh = v.view(B, T, n_heads, head_dim)

# B. Transpose: Swap T and n_heads to isolate heads in dim 1
# Shape becomes: [B, n_heads, T, head_dim]
q_mh = q_mh.transpose(1, 2)
k_mh = k_mh.transpose(1, 2)
v_mh = v_mh.transpose(1, 2)

print(f"Multi-Head Split:  {q_mh.shape}")    # [2, 8, 10, 64]

# C. Compute Attention (Independent Processing)
# F.sdpa treats dimensions [B, n_heads] as the "batch".
# Head 1 calculates scores totally independently of Head 2.
out_mh = F.scaled_dot_product_attention(q_mh, k_mh, v_mh)

# D. Concatenate (Merge Viewpoints)
# Transpose back: [B, T, n_heads, head_dim]
out_mh = out_mh.transpose(1, 2)

# Flatten last two dims: [B, T, C]
out_mh = out_mh.contiguous().view(B, T, C)

print(f"Multi-Head Merged: {out_mh.shape}")  # [2, 10, 512]


#### **1.3.2 Rotary Positional Embedding (RoPE)**

In the previous section, we discussed how Multi-Head Attention allows the model to capture complex relationships between tokens. However, the standard dot product used in attention has a major blind spot: it is permutation invariant. This means that if you shuffle the words in a sentence, the attention mechanism treats them exactly the same way. The model knows *what* the words are, but it does not inherently know *where* they are.

To address this, early Transformer models used **Absolute Positional Embeddings**. This method assigns a fixed or learned vector to each absolute position index (such as position 1 or position 5) and adds this vector to the input embedding. Later research introduced **Relative Positional Embeddings**. Instead of adding to the input, this approach modifies the attention score calculation to focus on the distance between two tokens.

The current standard for Large Language Models is **Rotary Positional Embedding (RoPE)** (https://arxiv.org/abs/2104.09864). This technique is designed to combine the advantages of both previous approaches. It maintains the implementation efficiency of absolute embeddings while effectively capturing the relational dependencies found in relative embeddings.

RoPE takes a different mathematical approach than its predecessors. It does not add a separate vector to the input. Instead, it applies a geometric rotation. This operation happens directly inside the Attention layer and is applied specifically to the **Query ($q$)** and **Key ($k$)** vectors.

To understand the mechanics, we look at the hidden state vector of size $d$. We treat this vector not as a single block of numbers, but as a sequence of $d/2$ independent pairs. Geometrically, we view each pair of numbers as **coordinates $(x, y)$ in a 2D plane**. To encode the position $m$ of a specific token, we simply rotate these coordinates by a specific angle derived from that position.

To formalize this, we construct a block-diagonal rotation matrix. If our model dimension $d$ is even, we define the transformation matrix $R_{\Theta, m}^d$ for a token at position $m$ as a sparse matrix populated by **$2 \times 2$ rotation blocks** along the diagonal. The full $d \times d$ matrix looks like this:

$$
R_{\Theta, m}^d = \begin{pmatrix}
\cos m\theta_0 & -\sin m\theta_0 & 0 & 0 & \cdots & 0 & 0 \\
\sin m\theta_0 & \cos m\theta_0 & 0 & 0 & \cdots & 0 & 0 \\
0 & 0 & \cos m\theta_1 & -\sin m\theta_1 & \cdots & 0 & 0 \\
0 & 0 & \sin m\theta_1 & \cos m\theta_1 & \cdots & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \cdots & \cos m\theta_{d/2-1} & -\sin m\theta_{d/2-1} \\
0 & 0 & 0 & 0 & \cdots & \sin m\theta_{d/2-1} & \cos m\theta_{d/2-1}
\end{pmatrix}
$$

It is crucial to understand the parameters defining the angle $m\theta_i$. Here, **$m$ represents the absolute position** of the token in the sequence (e.g., the 5th word has $m=5$). The term **$\theta_i$ represents the base frequency** for the $i$-th pair of dimensions. It is typically defined as $\theta_i = 10000^{-2i/d}$, where $i \in [0, 1, ..., d/2-1]$ is the index of the pair. This means that pairs at the beginning of the vector ($i=0$) have a large $\theta$ and rotate very quickly as position $m$ increases, while pairs at the end of the vector rotate very slowly. This **multi-scale rotation** allows the model to distinguish both close-range neighbors (via fast rotations) and long-range dependencies (via slow rotations).

The elegance of RoPE reveals itself when we compute the attention score. Recall that attention is driven by the dot product of a Query at position $m$ and a Key at position $n$. When we apply the rotary matrix to both vectors, we can mathematically trace how the positional information interacts:

$$
\begin{aligned}
\text{Score}(q_m, k_n) &= (R_{\Theta, m}^d q)^T (R_{\Theta, n}^d k) \\
&= q^T (R_{\Theta, m}^d)^T R_{\Theta, n}^d k \\
&= q^T R_{\Theta, -m}^d R_{\Theta, n}^d k \\
&= q^T R_{\Theta, (n-m)}^d k
\end{aligned}
$$

The derivation above demonstrates a critical property of this mechanism. By applying standard matrix operations, the individual absolute positions $m$ and $n$ effectively cancel each other out during the dot product calculation. The resulting attention score depends solely on the **relative distance $(n-m)$**.

This implies that RoPE naturally encodes the relative relationship between tokens rather than their specific index in the sentence, allowing the Transformer to enjoy the theoretical benefits of relative positioning while retaining the computational efficiency of absolute position methods.

In [ ]:
import torch

# --- Setup Dimensions ---
B, T, head_dim = 1, 10, 64  # Batch=1, Seq_Len=10, Dimension per head=64
# Note: RoPE is usually applied to the head dimension, not the full model dimension.

# Simulate a Query vector (before rotation)
q = torch.randn(B, T, head_dim)

# ==========================================
# 1. Precompute Frequencies (Thetas)
# ==========================================
# We process the vector in pairs. For a size 64 vector, we have 32 pairs.
# theta_i = 10000^(-2i/d)
i = torch.arange(0, head_dim, 2).float()
theta = 1.0 / (10000 ** (i / head_dim))  # Shape: [32]

# ==========================================
# 2. Compute Rotation Angles (m * Theta)
# ==========================================
# m: absolute position indices [0, 1, ..., T-1]
m = torch.arange(T).float()

# Outer product: matrix of angles for every position 'm' and frequency pair 'i'
# Shape: [Seq_Len, head_dim/2] -> [10, 32]
angles = torch.outer(m, theta)

# Prepare cos and sin for broadcasting
# Reshape to [1, T, head_dim/2, 1] to match the pair structure of Q later
cos_m = angles.cos().view(1, T, -1, 1)
sin_m = angles.sin().view(1, T, -1, 1)

# ==========================================
# 3. Apply Rotation (Geometric View)
# ==========================================
# Reshape Q to view it as independent pairs (x, y) in a 2D plane.
# Shape: [B, T, head_dim] -> [B, T, 32, 2]
q_pairs = q.view(B, T, -1, 2)

# Split coordinates
x = q_pairs[..., 0] # The even indices
y = q_pairs[..., 1] # The odd indices

# Apply the rotation formula derived from the matrix:
# [ x' ] = [ cos  -sin ] [ x ]
# [ y' ] = [ sin   cos ] [ y ]
x_rot = x * cos_m.squeeze(-1) - y * sin_m.squeeze(-1)
y_rot = x * sin_m.squeeze(-1) + y * cos_m.squeeze(-1)

# Reassemble the pairs and flatten back to original shape
q_rotated = torch.stack((x_rot, y_rot), dim=-1).flatten(2)

print(f"Original Q shape: {q.shape}")        # [1, 10, 64]
print(f"Rotated Q shape:  {q_rotated.shape}") # [1, 10, 64]
# The vector is now position-aware. The dot product (q_rot @ k_rot.T)
# will naturally encode relative distance (n-m).
print(f"Diff: ", q - q_rotated)

### **1.4 Exercise: Training a Tiny Language Model**

In this exercise, you will strip away the complexities of natural language and train a miniature Transformer to generate a specific numeric sequence (e.g., your Student ID). This will isolate and reinforce the mechanics of **Tokenization** and **Teacher Forcing**.

**Scenario**:
You are building a "Student ID Generator". Instead of learning English grammar, your model must learn the sequence of digits that makes up your ID. You will manually handle the conversion from strings to tensors and prepare the shifted inputs required for training.

**Requirements**:

#### **1. The "Numeric" Tokenizer**
Since our "language" consists only of digits, we do not need complex algorithms like BPE. You must implement a direct mapping strategy.
*   **Vocabulary**: Your vocabulary size is **12**.
    *   Digits: `'0'` through `'9'`.
    *   Special Tokens: `'<BOS>'` (Begin of Sequence) and `'<EOS>'` (End of Sequence).
*   **Implementation**: Create two dictionaries manually:
    1.  `token2id`: Maps characters/tokens to integers (e.g., `'5' -> 5`, `'<BOS>' -> 10`).
    2.  `id2token`: Maps integers back to characters.

#### **2. Data Preparation (Teacher Forcing)**
You will train on a single sample: your own **Student ID** (or an arbitrary number sequence like `"12345678"`).
*   **Tokenization**: Convert your ID string into a list of integers using your `token2id` map.
*   **Sequence Construction**:
    *   Prepend the `<BOS>` token ID to the start.
    *   Append the `<EOS>` token ID to the end.
*   **Input/Label Splitting**:
    Construct the `input_ids` and `target_ids` tensors for Teacher Forcing:
    1.  **Input**: The sequence *excluding* the last token (Model sees `<BOS>` to `Last Digit`).
    2.  **Target**: The sequence *excluding* the first token (Model predicts `First Digit` to `<EOS>`).
    *   *Format*: Convert these to `torch.LongTensor` with shape `(1, sequence_length)`.

#### **3. Model Architecture (Provided)**
We have provided a `TinyTransformer` class for you. It is a simplified GPT-style decoder with the following specs:
*   `vocab_size=12`
*   `embed_dim=16`
*   `num_heads=2`
*   `block_size=32` (Max sequence length)

#### **4. Execution**
*   **Loss Function**: Use `nn.CrossEntropyLoss`.
*   **Training Loop**:
    *   Run a loop for **50-100 iterations**.
    *   Since we are training on a single sample, we **want** the model to overfit. The loss should drop rapidly towards 0.
*   **Inference**:
    *   Use the provided `generate()` function.
    *   **Prompt**: Feed the model a single `<BOS>` token.
    *   **Goal**: The model should auto-regressively generate your exact Student ID followed by `<EOS>`.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ==========================================
# 1. The "Numeric" Tokenizer
# ==========================================

# Vocabulary configuration
VOCAB_SIZE = 12
SPECIAL_TOKENS = {'<BOS>': 10, '<EOS>': 11}

# TODO: Define the vocabulary mappings.
# Requirements:
# 1. '0' through '9' should map to indices 0-9.
# 2. '<BOS>' should map to 10, '<EOS>' should map to 11.
# 3. id2token should be the reverse mapping of token2id.
token2id = {str(i): i for i in range(10)}
token2id.update(SPECIAL_TOKENS)
id2token = {idx: tok for tok, idx in token2id.items()}

In [ ]:
# ==========================================
# 2. Data Preparation (Teacher Forcing)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Your Student ID (or arbitrary numeric sequence)
student_id_str = "12345678"

# TODO: Convert the string to a list of integers using token2id.
tokenized_sequence = [token2id[char] for char in student_id_str]

# TODO: Add special tokens.
# 1. Prepend <BOS> token ID.
# 2. Append <EOS> token ID.
full_sequence = [token2id["<BOS>"]] + tokenized_sequence + [token2id["<EOS>"]]

# TODO: Create Input and Target Tensors for Teacher Forcing.
# input_ids: The sequence excluding the last token (Model sees <BOS>...LastDigit).
# target_ids: The sequence excluding the first token (Model predicts FirstDigit...<EOS>).
# Note: Both must be converted to torch.LongTensor with shape (1, sequence_length).
input_tensor = torch.tensor([full_sequence[:-1]]).to(device)
target_tensor = torch.tensor([full_sequence[1:]]).to(device)

print(f"Input Tensor Shape: {input_tensor.shape if input_tensor is not None else 'None'}")
print(f"Target Tensor Shape: {target_tensor.shape if target_tensor is not None else 'None'}")


In [ ]:
# ==========================================
# 3. Model Architecture (Provided)
# ==========================================

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size=12, embed_dim=16, block_size=32, num_heads=2):
        super().__init__()
        self.block_size = block_size

        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(block_size, embed_dim)

        # Self-Attention Components
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Feed Forward Network
        self.fc_up = nn.Linear(embed_dim, embed_dim * 4)
        self.fc_down = nn.Linear(embed_dim * 4, embed_dim)
        self.gelu = nn.GELU()

        # Layer Norms
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

        # Output Head
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape # Batch, Time (Sequence Length)

        # 1. Embeddings
        tok_emb = self.token_embedding(idx)
        pos_emb = self.pos_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb

        # 2. Multi-Head Self-Attention Block
        residual = x
        x = self.ln1(x)

        # Projections
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot Product Attention with Causal Masking
        # is_causal=True ensures the model cannot look at future tokens
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Reassemble heads
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
        x = residual + self.out_proj(attn_out)

        # 3. Feed Forward Block
        residual = x
        x = self.ln2(x)
        x = self.fc_down(self.gelu(self.fc_up(x)))
        x = residual + x

        # 4. Final Logits
        logits = self.lm_head(x)
        return logits


# Initialize Model
model = TinyTransformer(vocab_size=VOCAB_SIZE)
print("Model initialized.")


In [ ]:
# ==========================================
# 4. Training Loop
# ==========================================

optimizer = optim.AdamW(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

print("Starting training...")

for step in range(100):
    optimizer.zero_grad()

    # TODO: Forward pass.
    # Feed the input_tensor into the model to get logits.
    logits = model(input_tensor)

    # TODO: Calculate Loss.
    # 1. Reshape logits to (Batch * Seq_Len, Vocab_Size).
    # 2. Reshape target_tensor to (Batch * Seq_Len).
    # 3. Compute CrossEntropyLoss between flattened logits and targets.
    loss = criterion(
        logits.view(-1, VOCAB_SIZE),
        target_tensor.view(-1)
    )

    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

print("Training complete.")

In [ ]:
# ==========================================
# 5. Inference (Generation)
# ==========================================

def generate(model, start_token="<BOS>", max_len=20):
    model.eval()  # Switch to evaluation mode

    # Initialize context with start token
    ctx = torch.tensor([[token2id[start_token]]], dtype=torch.long)

    with torch.no_grad():
        for _ in range(max_len):
            # Forward pass
            logits = model(ctx)

            # Get logits for the last token and pick the most likely one (Greedy)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            # Append prediction to current context
            ctx = torch.cat((ctx, next_token), dim=1)

            # Stop if <EOS> is generated
            if next_token.item() == token2id['<EOS>']:
                break

    # Decode result
    output_ids = ctx[0].tolist()
    decoded = [id2token[i] for i in output_ids if i not in (token2id['<BOS>'], token2id['<EOS>'])]
    return "".join(decoded)

# Run inference
generated_id = generate(model)
print(f"Generated Sequence: {generated_id}")

assert generated_id == student_id_str, "The model failed to memorize the ID!"

## Part 2: LLM Inference

This section shifts our focus from training a model to using it for text generation. We will explore the core mechanics of inference, which involves a step-by-step process where the model's own output becomes its next input.

Our goal is to understand the techniques that control what a model generates and how efficiently it does so.

- **Decoding Strategies**: We will examine methods for selecting the next token from the model's output probabilities. This includes moving beyond the simple greedy approach to explore sampling techniques like **Temperature** and **Top-K**, which manage the balance between coherence and creativity in the generated text.
- **KV Cache**: We will cover a critical optimization for accelerating text generation. You will learn how caching intermediate attention scores works and why it is essential for the performance of modern Large Language Models.



### **2.1 Decoding Strategies**

We have established the principles of the Transformer architecture and how the model predicts the next token during training. Now we face the core challenge of inference: converting the model's mathematical output into coherent human language.

At every step, the Large Language Model provides a probability distribution over the entire vocabulary—a set of `logits`. However, having these probabilities is not enough; we must decide how to strictly or creatively select the specific next token based on them. This decision process is what we call a decoding strategy.

In this section, we will implement the evolution of these strategies. We will start with **deterministic methods** like Greedy and Beam Search, which aim for the most probable outcome. Then, we will shift to **stochastic methods**, looking at how we can manipulate the probability distribution using Temperature, Top-k, and Top-p (Nucleus) sampling to introduce controlled randomness and generate text that is diverse rather than repetitive.

#### **2.1.1 Greedy Search**

<img width="75%" src="https://pic4.zhimg.com/v2-9db7d1ab898e3f01cb8022cf5461e82b_1440w.png">

Greedy Search is the most straightforward and intuitive decoding algorithm available. The mechanism is simple: at each time step, the model examines the probability distribution over the entire vocabulary and selects the single token with the highest probability. It does not consider future consequences or alternative paths; it simply commits to the most likely option available at that specific moment.

This approach is highly computationally efficient and easy to implement. By ensuring that the most probable word is always chosen, it guarantees a locally optimal decision at every step. However, this short-sighted nature is also its significant drawback. Because it never explores lower-probability paths that might lead to a better overall sequence, Greedy Search often fails to find the global optimum. The resulting text tends to be rigid and predictable, lacking the diversity and creativity inherent in natural language.

Furthermore, this method frequently leads to repetitive and tedious outputs. Since the model always picks the highest probability token, it can easily get stuck in loops. For example, given the input *“My favorite color is”*, a Greedy Search might generate: *“My favorite color is blue blue blue blue is blue and blue is my favorite color blue”*. As this example demonstrates, the algorithm repeatedly selects the word "blue" without adding any meaningful information, resulting in unnatural text.

In [ ]:
import torch
import torch.nn.functional as F

# --- Setup: Mock Vocabulary and Input ---
# A tiny vocabulary mapping indices to words
vocab = {0: "The", 1: "AI", 2: "writes", 3: "code", 4: "quickly", 5: "."}
vocab_size = len(vocab)

# Initial context sequence: ["The"]
# Shape: [batch_size=1, sequence_length=1]
input_ids = torch.tensor([[0]])

print(f"Initial Context: {[vocab[id.item()] for id in input_ids[0]]}\n")

# --- Greedy Search Implementation ---
num_new_tokens = 4

for step in range(num_new_tokens):
    # 1. Simulate Model Forward Pass
    # In practice: logits = model(input_ids).logits
    # Here: We generate deterministic random logits to simulate a model's output
    # Shape: [batch_size, seq_len, vocab_size]
    torch.manual_seed(step + 42) # Fixed seed to ensure consistent demo output
    logits = torch.randn(1, input_ids.shape[1], vocab_size)

    # 2. Extract Logits for the Next Token
    # We only care about the prediction for the last token in the sequence
    # Shape: [batch_size, vocab_size]
    next_token_logits = logits[:, -1, :]

    # 3. GREEDY STRATEGY: Argmax
    # Select the single token with the highest probability (logit)
    # No sampling, no randomness in selection
    next_token_id = torch.argmax(next_token_logits, dim=-1)

    # 4. Update Sequence (Autoregressive)
    # Append the predicted token to the input_ids for the next iteration
    input_ids = torch.cat([input_ids, next_token_id.unsqueeze(0)], dim=1)

    # (Optional) Visualization of the decision
    probs = F.softmax(next_token_logits, dim=-1)
    prob_val = probs[0, next_token_id].item()
    word = vocab[next_token_id.item()]
    print(f"Step {step+1}: Greedy chose '{word}' (probability: {prob_val:.4f})")

print(f"\nFinal Generated Sequence: {[vocab[id.item()] for id in input_ids[0]]}")

#### **2.1.2 Beam Search**

To mitigate the short-sightedness of Greedy Search, we introduce Beam Search. Instead of committing to a single path, this algorithm maintains multiple active hypotheses at every time step, allowing it to explore the search space more comprehensively. The number of hypotheses tracked simultaneously is defined by the **beam width** (denoted as $k$). By keeping the top $k$ most promising sequences based on their **cumulative probability**, Beam Search aims to find a sequence that is more globally optimal rather than just locally best.

The process operates iteratively. It begins by selecting the top $k$ most likely initial tokens as candidate sequences. In each subsequent step, the algorithm expands every current candidate by appending the top probable next tokens, temporarily creating a much larger set of new sequences. It then scores these expanded sequences and prunes the list, retaining only the $k$ sequences with the highest total scores. This cycle of expansion and selection continues until the sequences reach a maximum length or end with a termination token.

For example, consider generating the sentence *"the cat sat on the mat"* with a beam width of 2.

<img width="75%" src="https://pic4.zhimg.com/v2-cf720200ee9b254517e07fa9649c96ff_1440w.png">

The model might start with two candidates, "the" and "a". It then evaluates the next likely tokens for both. If the cumulative probability of "the cat" and "a cat" is higher than "the big" or "a mat", the latter are discarded—even if "big" was locally probable, the algorithm prioritizes the strength of the overall path. In this way, a path that starts slightly weaker but becomes stronger later can survive, unlike in Greedy Search.

While Beam Search strikes a better balance between exploration and utilization, it comes with trade-offs. A larger beam width explores more possibilities and generally yields better quality, but it linearly increases computational costs. It is also worth noting that if we set the beam width to 1, the algorithm behaves exactly like Greedy Search.

Despite its ability to produce more coherent text, Beam Search is not a universal cure for repetitiveness. It tends to favor "safe," high-probability sequences and can still suffer from repetition. For instance, given the input *“My favorite color is”* with a beam width of 3, the model might produce variations like *“My favorite color is blue because blue is a great color”* or *“My favorite color is blue, and I love blue clothes”*. While the model generates multiple options, they often remain structurally similar and repetitive, recycling the same words without introducing true diversity.

In [ ]:
import torch
import torch.nn.functional as F

# --- Configuration ---
beam_width = 3
max_steps = 5
vocab_size = 100
start_token = 0

# Mock Model: Simulates an LLM forward pass returning logits for the next token
def get_next_token_logits(sequence):
    # In a real scenario, this would be: model(sequence).logits[-1, :]
    torch.manual_seed(len(sequence)) # Fix seed for reproducibility in this example
    return torch.randn(vocab_size)

# --- Beam Search Implementation ---

# Initialization: List of (sequence_list, cumulative_log_prob)
# We use log-probabilities (score = 0.0) because probabilities multiply,
# but log-probabilities add, preventing numerical underflow.
beams = [([start_token], 0.0)]

print(f"Initial Beam: {beams}")

for step in range(max_steps):
    candidates = []

    # 1. Expansion: Process every currently active beam
    for seq, score in beams:
        logits = get_next_token_logits(seq)

        # Convert logits to log-probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Optimization: We only need to expand the top-k most likely next tokens
        # for this specific path, rather than the entire vocabulary.
        top_scores, top_indices = torch.topk(log_probs, beam_width)

        for i in range(beam_width):
            token_id = top_indices[i].item()
            token_score = top_scores[i].item()

            # Create new candidate path
            new_seq = seq + [token_id]
            # Update cumulative score (Add log_prob)
            new_score = score + token_score
            candidates.append((new_seq, new_score))

    # 2. Selection (Pruning): Rank ALL candidates from ALL expansions
    # Sort by score descending (higher log-prob is better) and keep only top k
    beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]

    # Visualization of the top candidate at this step
    print(f"Step {step+1}: Best Beam Score: {beams[0][1]:.4f} | Seq: {beams[0][0]}")

# Final Result
best_sequence = beams[0][0]
print(f"\nFinal Selected Sequence: {best_sequence}")


#### **2.1.3 Random Sampling & Temperature**

<img width="75%" src="https://pic4.zhimg.com/v2-8c85af2decfd1586e1e37b05f8168ebf_1440w.png">

Human-generated text is characterized by a unique probability distribution with distinct peaks, reflecting the natural variability and creativity of human thought. In contrast, traditional algorithms like Beam Search tend to produce uniform, "safe," and often repetitive outputs. To overcome this limitation and mimic the richness of human language, we turn to sampling-based decoding techniques. Instead of deterministically selecting the maximum probability, these methods generate text by randomly sampling from the model's predicted distribution, leading to more diverse and unpredictable outcomes.

The most fundamental form of this approach is **Random Sampling**. In this method, the next word is chosen stochastically based on its probability mass. For instance, if a word has a 10% probability, it has a 1 in 10 chance of being selected, meaning the model is not bound to the single most likely token. This mechanism allows the generation process to occasionally select rare words, introducing a level of surprise and variation that deterministic methods cannot achieve.

However, pure random sampling has a significant flaw located in the "long tail" of the probability distribution. The long tail refers to the vast number of vocabulary words to which the model assigns very low probabilities. While sampling increases diversity, it also increases the risk of selecting these low-confidence tokens. When the model over-samples from this tail, the generated text quality deteriorates, resulting in content that may become incoherent, grammatically incorrect, or logically confused.

To control this trade-off between diversity and correctness, we use a hyperparameter called **Temperature**. Temperature works by scaling the logits before they are converted into probabilities. A low temperature ($< 1.0$) sharpens the distribution, making high-probability words even more likely and suppressing the long tail (acting more like Greedy Search). Conversely, a high temperature ($> 1.0$) flattens the distribution, giving the long tail a higher chance of being picked. This allows us to dynamically tune the model's behavior: lower temperatures for factual, precise answers, and higher temperatures for creative, diverse storytelling.

In [ ]:
import torch
import torch.nn.functional as F

# 1. Simulate raw logits from the LLM's final layer
# Context: "The quick brown fox jumps over the..."
# Vocabulary candidates: ["dog", "lazy", "fence", "moon", "pizza"]
vocab = ["dog", "lazy", "fence", "moon", "pizza"]
logits = torch.tensor([4.5, 3.0, 1.5, -0.5, -2.0])

def sample_with_temperature(logits, temperature):
    """
    Demonstrates the effect of temperature on probability distribution and sampling.
    """
    print(f"\n--- Temperature: {temperature} ---")

    # Step 1: Temperature Scaling
    # T < 1.0: Sharpens distribution (peaks get higher, tail gets lower).
    # T > 1.0: Flattens distribution (peaks get lower, tail gets higher).
    scaled_logits = logits / temperature

    # Step 2: Convert to Probabilities (Softmax)
    probs = F.softmax(scaled_logits, dim=-1)

    # Visualizing the shift in probability mass
    print(f"Probs: {[f'{p:.3f}' for p in probs.tolist()]}")

    # Step 3: Random Sampling
    # torch.multinomial samples indices based on the input probabilities.
    # This introduces the stochastic element (unlike argmax).
    next_token_idx = torch.multinomial(probs, num_samples=1)

    print(f"Selected: '{vocab[next_token_idx.item()]}'")

# 2. Execute with different temperatures

# Case A: Low Temperature (Conservative)
# The model becomes very confident in the top choice ("dog").
# The "long tail" (moon, pizza) is effectively suppressed.
sample_with_temperature(logits, temperature=0.5)

# Case B: Standard Random Sampling (T=1.0)
# Reflects the model's raw predictions. "lazy" has a decent chance.
sample_with_temperature(logits, temperature=1.0)

# Case C: High Temperature (Creative / Chaotic)
# The distribution flattens. Rare words like "pizza" gain significant probability.
sample_with_temperature(logits, temperature=2.0)


#### **2.1.4 Top-k Sampling**

<img width="75%" src="https://pic1.zhimg.com/v2-bbf6239a44e876cb22fa29f0b770a1a0_1440w.png">

To mitigate the risks associated with the long tail in pure random sampling, we introduce a technique known as Top-k Sampling. The core idea is simple yet effective: at each generation step $t$, instead of sampling from the entire vocabulary, we restrict the pool of candidates to only the $k$ words with the highest probabilities. All other words are effectively filtered out, and the probability mass is redistributed among these top $k$ candidates before sampling.

This method strikes a practical balance by ensuring that the model only considers reasonable options while still preserving the stochastic nature of generation. By introducing this controlled randomness, Top-k sampling allows the model to explore alternative narrative paths, resulting in output that feels more diverse, creative, and closer to human-generated text than deterministic methods. The value of $k$ acts as a direct control knob for this behavior: setting $k=1$ reverts the process to Greedy Search, while setting $k$ to the size of the entire vocabulary ($V$) is equivalent to pure Random Sampling.

However, Top-k sampling is not without its flaws. Compared to Beam Search, it can sometimes yield less coherent text because it doesn't look ahead. More importantly, selecting the right value for $k$ is challenging because the "correct" number of plausible next words varies depending on the context. A static $k$ might be too small in some situations, cutting off valid creative choices, or too large in others, allowing the model to sample irrelevant or nonsensical words from the tail of the distribution.

In [ ]:
import torch
import torch.nn.functional as F

# Setup: Simulate raw logits for the next token (Batch=1, Vocab=10)
# In practice: logits = model(input_ids)[:, -1, :]
logits = torch.randn(1, 10)
k = 3

print(f"Raw Logits: {logits[0].tolist()}")

# --- Top-k Sampling Implementation ---

# 1. Find the threshold value
# We retrieve the k-th highest logit. Any logit below this value will be masked.
# topk returns (values, indices). We take the last value (smallest of the top k).
top_k_values, _ = torch.topk(logits, k)
threshold = top_k_values[:, -1].unsqueeze(-1)  # Shape: [1, 1]

# 2. Masking (Filtering)
# Set logits below the threshold to negative infinity.
# This ensures their probability becomes 0 after Softmax.
filtered_logits = torch.where(
    logits < threshold,
    torch.tensor(float('-inf')),
    logits
)

# 3. Re-normalization (Redistribute probability mass)
# Apply softmax to the filtered logits. The mass is now shared only among top-k.
probs = F.softmax(filtered_logits, dim=-1)

# 4. Sampling
# Sample from the new distribution (stochastic selection among the top k).
next_token = torch.multinomial(probs, num_samples=1)

# --- Output Visualization ---
print(f"\nK={k} Threshold: {threshold.item():.4f}")
print("Probabilities after Top-k filtering:")
for i, p in enumerate(probs[0]):
    status = "Candidate" if p > 0 else "Filtered"
    print(f"  Token {i}: {p:.4f} ({status})")

print(f"\nSelected Next Token ID: {next_token.item()}")


#### **2.1.5 Top-p (Nucleus) Sampling**

<img width="75%" src="https://pic3.zhimg.com/v2-cb197cb03300c3f6ce442e1e908aebae_1440w.png">

Top-p sampling, also known as Nucleus Sampling, is a sophisticated variant of decoding that aims to strike a precise balance between the diversity of the text and its coherence. Unlike Top-k, which rigidly selects a fixed number of candidates regardless of the context, Top-p works by dynamically truncating the probability distribution. At each time step, the model sorts the vocabulary by probability and retains only the smallest set of top words whose **cumulative probability** exceeds a chosen threshold $p$. The model then samples exclusively from this "nucleus" of probable words.

The primary advantage of this approach is its adaptability. By focusing on the cumulative probability, the size of the candidate pool changes naturally based on the model's confidence. If the model is unsure (a flat distribution), the nucleus expands to include many words, allowing for variety. If the model is confident (a sharp distribution), the nucleus shrinks to just a few dominant words, ensuring coherence. This dynamic adjustment leads to text that effectively mimics the flow and structure of human-generated language.

The parameter $p$ acts as a direct control knob for the trade-off between creativity and logic. For instance, setting $p = 0.9$ allows the model to consider the top 90% of the probability mass, filtering out only the unlikely "long tail" while maintaining a broad range of creative options. Lowering the value (e.g., to $p = 0.5$) forces the model to become stricter, selecting from a much narrower set of high-confidence words, which increases coherence but reduces variety. Setting $p = 1.0$ includes the entire vocabulary, effectively reverting the process to pure random sampling with maximum creativity but a higher risk of incoherence.

However, Top-p sampling is not without its limitations. The choice of $p$ is critical; if the threshold is set too low, the output may become repetitive or overly generic, while setting it too high can reintroduce the noise of unlikely words. Additionally, because the algorithm requires sorting the vocabulary and calculating cumulative sums at every single step, it can be more computationally intensive than simpler strategies like Top-k sampling.

In [ ]:
import torch
import torch.nn.functional as F

# Simulate a distribution
logits = torch.randn((1, 10))
top_p = 0.85  # The cumulative probability threshold

# 1. Sort logits in descending order
# We need to sort to calculate the cumulative sum correctly
sorted_logits, sorted_indices = torch.sort(logits, descending=True)

# 2. Calculate cumulative probabilities
sorted_probs = F.softmax(sorted_logits, dim=-1)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

# 3. Create the Nucleus Mask
# Identify tokens where cumulative probability exceeds top_p
sorted_indices_to_remove = cumulative_probs > top_p

# CRITICAL STEP: Shift the mask right by 1.
# We want to KEEP the first token that crosses the threshold, so we shift
# the "remove" mask to start after that token.
sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
sorted_indices_to_remove[..., 0] = 0 # Always keep the most probable token

# 4. Filter logits (Apply mask)
# Set logits of non-nucleus tokens to negative infinity
sorted_logits[sorted_indices_to_remove] = float('-inf')

# 5. Sample from the Nucleus
# Re-normalize probabilities (softmax) based only on the nucleus
nucleus_probs = F.softmax(sorted_logits, dim=-1)
sample_idx_in_sorted = torch.multinomial(nucleus_probs, num_samples=1)

# 6. Map back to original vocabulary index
next_token_id = sorted_indices.gather(1, sample_idx_in_sorted)

# --- Visualization for Tutorial ---
print(f"Original Probs:   {F.softmax(logits, dim=-1).numpy().round(3)}")
print(f"Sorted Probs:     {sorted_probs.numpy().round(3)}")
print(f"Cumulative Probs: {cumulative_probs.numpy().round(3)}")
print(f"Nucleus Mask:     {~sorted_indices_to_remove.numpy()} (True means kept)")
print(f"Selected Token:   {next_token_id.item()}")


### **2.2 KV Cache**

We have just examined *how* to decide which token comes next using various decoding strategies. However, knowing *what* to generate is only half the battle; we must also generate it *efficiently*.

Text generation is an autoregressive process, meaning the model produces output one token at a time. In a naive implementation, this can be painfully slow, as the computational cost grows significantly as the generated text gets longer. Without optimization, generating a long paragraph could take a prohibitively long time, even on powerful hardware.

In this section, we will explore the **KV Cache** (Key-Value Cache), the de facto standard optimization technique used in almost every production LLM inference engine. We will analyze the computational bottleneck inherent in Transformers, explain the mechanism of caching intermediate states to avoid redundant calculations, and dive into the practical details of managing tensor shapes during the generation loop.

#### **2.2.1 The Autoregressive Bottleneck**

To understand why optimization is necessary, we must first revisit the computational nature of autoregressive text generation. As we have seen, Large Language Models generate text sequentially, one token at a time. The output of step $t$ is appended to the input sequence to become the input for step $t+1$.

In a naive implementation, this process is incredibly inefficient. Suppose we are generating the 100th token of a sentence. To do this, we feed all 99 previous tokens into the model. The model processes these 99 tokens to compute their internal representations (including their Key and Value vectors) just to predict the 100th one. When we move to generate the 101st token, we feed in the now 100-token sequence. The model then re-calculates the exact same representations for the first 99 tokens all over again, even though they have not changed.

This redundant re-computation creates a significant bottleneck. The attention mechanism, specifically, has a quadratic complexity with respect to sequence length. If we re-process the entire history for every new token generated, the computational cost grows explosively as the sequence gets longer. This "autoregressive bottleneck" makes real-time text generation with long contexts prohibitively slow and expensive without further optimization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NaiveTransformer(nn.Module):
    def __init__(self, vocab_size=100, d_model=16, n_heads=2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, 1024, d_model))
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.output = nn.Linear(d_model, vocab_size)
        self.n_heads = n_heads

    def forward(self, x):
        # x shape: (batch, seq_len)
        b, t = x.shape
        h = self.token_emb(x) + self.pos_emb[:, :t, :]

        # Self-Attention using Scaled Dot Product Attention (SDPA)
        qkv = self.qkv(h).reshape(b, t, 3, self.n_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Causal mask is applied here
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(b, t, -1)
        return self.output(attn_out)

# Simulation of the Autoregressive Bottleneck
model = NaiveTransformer()
input_ids = torch.tensor([[1, 2, 3]]) # Initial prompt
generated_len = 5

print(f"Initial sequence: {input_ids.tolist()}")

for i in range(generated_len):
    # --- THE BOTTLENECK ---
    # Every step, we pass the ENTIRE sequence through the model.
    # If input_ids has length 100, we re-calculate K and V for all 100 tokens
    # just to get the logit for the 101st token.
    logits = model(input_ids)

    # We only care about the last token's prediction
    next_token_logits = logits[:, -1, :]
    next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

    # Append and repeat: The sequence grows, and so does the redundant computation
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(f"Step {i+1}: Input length = {input_ids.shape[1]}, "
          f"Redundantly processed tokens = {input_ids.shape[1]-1}")

print(f"Final sequence: {input_ids.tolist()}")


#### **2.2.2 Caching Keys and Values**

The solution to the autoregressive bottleneck lies in a classic computer science strategy: trading memory for compute. We can observe that the weights of the model are frozen during inference. Consequently, for any given token in the past history, its projection into Key ($K$) and Value ($V$) vectors depends only on its own embedding and the model's weights. These vectors do not change when we add a new token to the end of the sequence.

This observation leads to the **KV Cache** optimization. Instead of discarding the Key and Value vectors after each step, we store them in memory. When the model generates the next token at step $t$, we only need to perform the forward pass for this single new token. We compute its specific Query ($q_t$), Key ($k_t$), and Value ($v_t$).

Then, rather than re-computing the attention for the entire history, we simply retrieve the cached Keys and Values ($K_{past}$ and $V_{past}$) from memory. We append the new $k_t$ and $v_t$ to this cache and use the combined set to perform the attention operation. Crucially, we do **not** cache the Query vectors. The Query represents the "current question" the model is asking (i.e., "what comes next?"), which is specific to the current token and does not need to be preserved for future steps. This technique reduces the redundant computation from quadratic to linear time per step, dramatically accelerating inference.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KVCacheTransformer(nn.Module):
    def __init__(self, vocab_size=100, d_model=16, n_heads=2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, 1024, d_model))
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.output = nn.Linear(d_model, vocab_size)
        self.n_heads = n_heads

    def forward(self, x, past_kv=None, pos_offset=0):
        # x shape: (batch, 1) during generation after the first step
        b, t = x.shape
        # Use pos_offset to get the correct positional embedding for the new token
        h = self.token_emb(x) + self.pos_emb[:, pos_offset : pos_offset + t, :]

        qkv = self.qkv(h).reshape(b, t, 3, self.n_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2] # current token's Q, K, V

        if past_kv is not None:
            past_k, past_v = past_kv
            # Concatenate past Keys/Values with current ones (The Cache Update)
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)

        # Save current K, V for the next step
        present_kv = (k, v)

        # Attention: Query is only for the new token, but Keys/Values cover the whole history
        # is_causal=False because we handle the 'history' via concatenation
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(b, t, -1)
        return self.output(attn_out), present_kv

# Simulation of Efficient Generation with KV Cache
model = KVCacheTransformer()
input_ids = torch.tensor([[1, 2, 3]]) # Initial prompt
generated_len = 5
kv_cache = None

print(f"Initial sequence: {input_ids.tolist()}")

# 1. Prefill Phase: Process the initial prompt to build the first cache
logits, kv_cache = model(input_ids, past_kv=None, pos_offset=0)
next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
full_sequence = torch.cat([input_ids, next_token], dim=-1)

# 2. Decoding Phase: Process only ONE token at a time
for i in range(generated_len - 1):
    # Only pass the LAST generated token into the model
    current_token = full_sequence[:, -1:]

    # The model only computes Q, K, V for this 1 token!
    logits, kv_cache = model(current_token, past_kv=kv_cache, pos_offset=full_sequence.shape[1]-1)

    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    full_sequence = torch.cat([full_sequence, next_token], dim=-1)

    print(f"Step {i+1}: Input length = 1 (Efficient!), "
          f"Cache size (K/V) = {kv_cache[0].shape[-2]} tokens")

print(f"Final sequence: {full_sequence.tolist()}")


#### **2.2.3 Cache Management and Tensor Shapes**

Implementing KV Cache requires careful management of tensor dimensions and state updates. This is the bridge between the theoretical concept and the actual code you will write. Unlike a standard forward pass where inputs are static, inference with caching involves two distinct phases: the **Prefill** phase and the **Decoding** phase.

During the **Prefill phase**, we process the user's entire input prompt at once. At this stage, we compute the Keys and Values for all prompt tokens in parallel and store them as the initial state of our cache. The shape of this cache tensor is typically `(Batch_Size, Num_Heads, Sequence_Length, Head_Dimension)`.

Once we enter the **Decoding phase**, the process changes to sequential updates. In each iteration, the input to the model is just the single most recently generated token—not the full sentence. We compute the $K$ and $V$ for this single token, which will have a sequence length of 1. We then concatenate these new vectors along the `Sequence_Length` dimension of our existing cache tensor.

Managing these shapes correctly is critical. You must ensure that your attention mechanism can dynamically handle a Query of length 1 interacting with Keys and Values of length $N$ (where $N$ is the total history length). This mismatch in sequence length between Query and Key/Value is the signature characteristic of cached inference.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KVManagedTransformer(nn.Module):
    def __init__(self, vocab_size=100, d_model=16, n_heads=2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, 1024, d_model))
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.output = nn.Linear(d_model, vocab_size)
        self.n_heads = n_heads

    def forward(self, x, past_kv=None, pos_offset=0):
        # x shape: (B, L_new) -> L_new is prompt_len (Prefill) or 1 (Decoding)
        b, t = x.shape
        h = self.token_emb(x) + self.pos_emb[:, pos_offset : pos_offset + t, :]

        # Project and split to (3, B, n_heads, L_new, head_dim)
        qkv = self.qkv(h).reshape(b, t, 3, self.n_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # --- Cache Management ---
        if past_kv is not None:
            # Concatenate along the Sequence Length dimension (dim=-2)
            k = torch.cat([past_kv[0], k], dim=-2)
            v = torch.cat([past_kv[1], v], dim=-2)
        present_kv = (k, v)

        # Shape Check: Q is (B, heads, L_new, d), K/V are (B, heads, L_total, d)
        # SDPA handles the L_new vs L_total mismatch automatically
        # During Decoding, L_new=1, L_total=history_len + 1
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=(t > 1))

        attn_out = attn_out.permute(0, 2, 1, 3).reshape(b, t, -1)
        return self.output(attn_out), present_kv

# --- Execution Flow ---
model = KVManagedTransformer()
prompt = torch.tensor([[10, 20, 30, 40]]) # Length 4
generated_tokens = []

# PHASE 1: PREFILL
# Process the entire prompt to "warm up" the cache
print(f"--- Prefill Phase (Input length: {prompt.shape[1]}) ---")
logits, kv_cache = model(prompt, past_kv=None, pos_offset=0)
next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
generated_tokens.append(next_token)

# PHASE 2: DECODING
# Process only the single last token, appending to the cache each time
print(f"--- Decoding Phase ---")
for i in range(3):
    curr_input = generated_tokens[-1]
    # Current sequence length for pos_emb is (prompt_len + number of tokens generated so far)
    offset = prompt.shape[1] + i

    logits, kv_cache = model(curr_input, past_kv=kv_cache, pos_offset=offset)

    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated_tokens.append(next_token)

    print(f"Step {i+1}: Input Shape {curr_input.shape} | Cache K Shape {kv_cache[0].shape}")

# Final Result
full_out = torch.cat([prompt, torch.cat(generated_tokens, dim=-1)], dim=-1)
print(f"Full Sequence: {full_out.tolist()}")


### **2.3 Exercise: KV Cache Implementation**

In the previous exercise, you trained a model to generate a sequence. You may have noticed that our `generate` function was inefficient. At each step, it re-processed the *entire* sequence of previously generated tokens to predict just one new token. This is the auto-regressive bottleneck we discussed.

In this exercise, you will implement the **KV Cache** mechanism to dramatically speed up this process. Instead of recomputing everything, you will cache the Key (K) and Value (V) tensors from previous steps and reuse them.

**Scenario**:
We will continue with our "Student ID Generator". You will use the exact same `TinyTransformer` model that you just trained. The goal is to modify its inference logic—without retraining—to make it much more efficient.

**Important**: Your Google Colab runtime may have timed out or been reset. Before starting this exercise, please go back and **re-run all the code cells from Exercise 1.4** to ensure the `model`, `token2id`, and `id2token` variables are initialized and the model is trained.

**Requirements**:

#### **1. A `forward` Method with KV Cache Support**
You will implement a new Python function that can serve as a replacement for the model's original `forward` method. This new function will accept an optional `kv_cache`.

*   **Function Signature**: `forward_with_cache(self, idx, kv_cache=None)`
*   **Logic**:
    1.  **Input `idx`**: Unlike the training `forward` pass, `idx` will now only contain the **newest token** (shape `(B, 1)`), except for the very first step.
    2.  **Cache Handling**:
        *   If `kv_cache` is `None` (i.e., the first generation step), initialize it as a tuple of empty tensors for keys and values.
        *   If a `kv_cache` is provided, unpack it.
    3.  **K, V Computation**:
        *   Calculate the Key and Value vectors **for the new input token `idx` only**.
        *   Append these new `k` and `v` vectors to the `k_cache` and `v_cache` from the previous step.
    4.  **Attention**:
        *   Calculate the Query vector (`q`) for the new token.
        *   Perform scaled dot-product attention using the new `q` and the **full, updated** `k_cache` and `v_cache`.
    5.  **Output**: The function must return two things:
        *   The `logits` for the current time step.
        *   The `updated_kv_cache` to be used in the next step.

We will provide the code to "monkey-patch" this new function onto your existing trained model instance, effectively replacing its forward pass for inference: `model.forward = forward_with_cache.__get__(model, TinyTransformer)`.

#### **2. An Optimized `generate` Function**
Next, you will write a new `generate_with_cache` function that leverages your new `forward` method.

*   **Initialization**: Start with a `<BOS>` token and an empty `kv_cache = None`.
*   **Generation Loop**: In each iteration of the loop, you must:
    1.  Call your new `forward_with_cache` method, passing the **current token** and the **current `kv_cache`**.
    2.  Receive the `logits` and the `updated_kv_cache` from the function call.
    3.  Store the `updated_kv_cache` to be passed into the *next* iteration.
    4.  Determine the next token from the `logits` (e.g., using `argmax`).
    5.  Repeat until an `<EOS>` token is generated or the maximum length is reached.
*   **Goal**: The function should produce the exact same output as the original `generate` function but perform far fewer computations.

In [ ]:
import torch
import torch.nn.functional as F
import time
import types

# =================================================================================
# IMPORTANT: Before running this cell, please re-run all the code from Exercise 1.4
# to ensure the 'model', 'token2id', and 'id2token' variables are defined and the
# model is trained.
# =================================================================================

# ==========================================
# 1. A `forward` Method with KV Cache Support
# ==========================================

def forward_with_cache(self, idx, kv_cache=None):
    B, T = idx.shape

    # TODO: If kv_cache is None (first step), initialize it as a tuple of two empty tensors.
    # The shape for k_cache and v_cache should be (B, num_heads, 0, head_dim).
    # Otherwise, unpack the provided kv_cache.
    if kv_cache is None:
        k_cache, v_cache = None, None
    else:
        k_cache, v_cache = kv_cache

    # TODO: Get the starting position for the positional embedding.
    # This should be the length of the sequence already in the cache (k_cache.shape[2]).
    pos_start = None

    # --- The rest of the model logic is almost identical ---

    pos_emb = self.pos_embedding(torch.arange(pos_start, pos_start + T, device=idx.device))
    tok_emb = self.token_embedding(idx)
    x = tok_emb + pos_emb

    residual = x
    x = self.ln1(x)

    q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    k_new = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    v_new = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    # TODO: Update the cache by concatenating the new k and v vectors with the existing cache.
    # The concatenation should happen along the sequence length dimension (dim=2).
    k = None
    v = None

    # Perform attention with the full key and value sequences
    attn_out = F.scaled_dot_product_attention(q, k, v)

    attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
    x = residual + self.out_proj(attn_out)

    residual = x
    x = self.ln2(x)
    x = self.fc_down(self.gelu(self.fc_up(x)))
    x = residual + x

    logits = self.lm_head(x)

    # TODO: Return the logits and the updated cache tuple (k, v).
    return None, None

# "Monkey-patch" the new forward method onto our existing model instance.
# This replaces the model's original .forward() with our new, cache-aware version.
model.forward = types.MethodType(forward_with_cache, model)
print("Model's forward method has been updated to support KV cache.")

In [ ]:
# ==========================================
# 2. An Optimized `generate` Function
# ==========================================

def generate_with_cache(model, start_token="<BOS>", max_len=20):
    model.eval()

    # TODO: Initialize the context 'ctx'. It should be a tensor containing just the start_token ID.
    # Shape: (1, 1)
    ctx = None

    # TODO: Initialize the kv_cache to None before the loop starts.
    kv_cache = None

    generated_ids = []
    with torch.no_grad():
        for _ in range(max_len):
            # TODO: Perform a forward pass.
            # Pass the current context (ctx) and the current kv_cache.
            # The model will return logits and the updated_kv_cache.
            logits, updated_kv_cache = None, None

            # TODO: Update the kv_cache for the next iteration.
            kv_cache = None

            # Greedy decoding: get the most likely next token from the logits
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            # Stop if <EOS> is generated
            if next_token.item() == token2id['<EOS>']:
                break

            generated_ids.append(next_token.item())

            # TODO: Update the context (ctx) for the next iteration.
            # It should be the newly generated token.
            ctx = None

    # Decode the generated IDs into a string
    decoded = "".join([id2token[i] for i in generated_ids])
    return decoded

# --- Verification ---
cached_output = generate_with_cache(model)
print(f"\nKV cache generation output: {cached_output}")

In [ ]:
def original_forward(self, idx):
    B, T = idx.shape # Batch, Time (Sequence Length)

    # 1. Embeddings
    tok_emb = self.token_embedding(idx)
    pos_emb = self.pos_embedding(torch.arange(T, device=idx.device))
    x = tok_emb + pos_emb

    # 2. Multi-Head Self-Attention Block
    residual = x
    x = self.ln1(x)

    # Projections
    q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    # Scaled Dot Product Attention with Causal Masking
    # is_causal=True ensures the model cannot look at future tokens
    attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # Reassemble heads
    attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
    x = residual + self.out_proj(attn_out)

    # 3. Feed Forward Block
    residual = x
    x = self.ln2(x)
    x = self.fc_down(self.gelu(self.fc_up(x)))
    x = residual + x

    # 4. Final Logits
    logits = self.lm_head(x)
    return logits

In [ ]:
%%timeit
model.forward = types.MethodType(original_forward, model)
generated_id = generate(model)
# print(f"Without KVCache - Generated Sequence: {generated_id}")

In [ ]:
%%timeit
model.forward = types.MethodType(forward_with_cache, model)
generated_id = generate_with_cache(model)
# print(f"With KVCache - Generated Sequence: {generated_id}")